# 03 · ABC-XYZ segmentation
**Business question:** which items deserve the most planning attention?

- **ABC** = value: items ranked by revenue over the last 52 weeks. A = top 80% of revenue, B = next 15%, C = last 5%.
- **XYZ** = predictability: coefficient of variation (CV = standard deviation ÷ mean) of weekly units. X ≤ 0.5 (stable), Y ≤ 1.0, Z > 1.0 (erratic).

*IBP comparison: the same segmentation planners use to decide which items get automated statistical forecasts and which need manual review.*

> **Run one notebook at a time.** DuckDB allows only one open connection to `practice.duckdb`. Before opening another notebook, use *Kernel → Shut Down Kernel* in this one.

In [ ]:
import duckdb

# Connect to the local DuckDB file built in 00_setup_and_load
con = duckdb.connect('../data/practice.duckdb')

# Memory settings: needed for the 59M-row sales_daily table on a laptop
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

## 1. Item revenue over the last 52 weeks

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE item_revenue_52w AS
SELECT item_id,
       dept_id,
       cat_id,
       SUM(revenue) AS total_revenue
FROM sales_daily
WHERE date BETWEEN '2015-05-25' AND '2016-05-22'   -- last 52 full weeks
GROUP BY item_id, dept_id, cat_id                  -- grain: item (all stores combined)
""")

In [ ]:
# Check: all 3,049 items present and none with zero revenue
con.sql("""
SELECT COUNT(*)           AS items,
       SUM(total_revenue) AS revenue_52w,
       SUM(CASE WHEN total_revenue IS NULL OR total_revenue = 0 THEN 1 ELSE 0 END) AS items_no_revenue
FROM item_revenue_52w
""").df()

,items,revenue_52w,items_no_revenue
0,3049,45041697.91,0.0


## 2. ABC classification
Rank items by revenue, compute the running share of total revenue, then cut at 80% and 95%.
The CTE is needed because the class is based on a window result (running total).

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE item_abc AS
WITH item_running AS (
    SELECT item_id,
           dept_id,
           cat_id,
           total_revenue,
           -- item_id breaks ties, so the ranking is always the same
           ROW_NUMBER() OVER (ORDER BY total_revenue DESC, item_id) AS rank_revenue,
           SUM(total_revenue) OVER (ORDER BY total_revenue DESC, item_id
                                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_tot_rev,
           SUM(total_revenue) OVER () AS tot_rev_all
    FROM item_revenue_52w
)
SELECT item_id,
       dept_id,
       cat_id,
       total_revenue,
       rank_revenue,
       ROUND(running_tot_rev * 100.0 / tot_rev_all, 1) AS cum_perc_rev,   -- rounded for display only
       CASE WHEN running_tot_rev * 100.0 / tot_rev_all <= 80 THEN 'A'      -- classify on the unrounded value
            WHEN running_tot_rev * 100.0 / tot_rev_all <= 95 THEN 'B'
            ELSE 'C'
       END AS abc_rev
FROM item_running
""")

In [ ]:
# ABC summary: share of items vs share of revenue
con.sql("""
SELECT abc_rev,
       COUNT(*)                         AS no_of_items,
       ROUND(COUNT(*) / 3049 * 100, 1)  AS perc_items,
       SUM(total_revenue)               AS total_rev_class_level,
       ROUND(SUM(total_revenue) / SUM(SUM(total_revenue)) OVER () * 100.0, 1) AS perc_tot_rev
FROM item_abc
GROUP BY abc_rev
ORDER BY abc_rev
""").df()

,abc_rev,no_of_items,perc_items,total_rev_class_level,perc_tot_rev
0,A,1318,43.2,36028172.28,80.0
1,B,943,30.9,6760602.99,15.0
2,C,788,25.8,2252922.64,5.0


**Findings: ABC**
- A items: 1,318 items (43%) bring 80% of revenue. B: 943 items (31%), 15%. C: 788 items (26%), 5%.
- The A class is larger than the textbook "20% of items", because M5 is grocery and household goods with many mid-value items rather than a few hero SKUs.

## 3. XYZ classification (demand variability)

In [ ]:
# Weekly units per item over the last 52 weeks
con.sql("""
CREATE OR REPLACE TABLE item_weekly_units_52w AS
SELECT item_id,
       wm_yr_wk,
       SUM(units) AS total_units
FROM sales_daily
WHERE date BETWEEN '2015-05-23' AND '2016-05-20'
GROUP BY item_id, wm_yr_wk                  -- grain: item × week
""")

In [ ]:
# CV = standard deviation / mean of weekly units
con.sql("""
CREATE OR REPLACE TABLE item_xyz AS
SELECT item_id,
       AVG(total_units)    AS avg_weekly_units,
       STDDEV(total_units) AS std_dev_units,
       ROUND(STDDEV(total_units) / AVG(total_units), 2) AS cv_items,      -- rounded for display only
       CASE WHEN STDDEV(total_units) / AVG(total_units) <= 0.5 THEN 'X'   -- classify on the unrounded value
            WHEN STDDEV(total_units) / AVG(total_units) <= 1.0 THEN 'Y'
            ELSE 'Z'
       END AS xyz_items
FROM item_weekly_units_52w
GROUP BY item_id
""")

In [ ]:
# XYZ summary
con.sql("""
SELECT xyz_items,
       COUNT(*) AS no_of_items,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS perc_of_items
FROM item_xyz
GROUP BY xyz_items
ORDER BY xyz_items
""").df()

,xyz_items,no_of_items,perc_of_items
0,X,2364,77.5
1,Y,584,19.2
2,Z,101,3.3


**Findings: XYZ**
- 78% of items are X (stable), 19% Y and only 3% Z, once demand is summed across all 10 stores.
- Aggregating across stores smooths demand. The same items would look far more erratic at store level, which is where replenishment actually happens.

## 4. ABC-XYZ matrix
Join the two classifications on `item_id`. Both tables are one row per item, so `item_id` alone is the full key and there is no fan-out.

In [ ]:
con.sql("""
SELECT a.abc_rev || x.xyz_items AS abc_xyz_class,     -- || joins text: 'A' || 'X' = 'AX'
       COUNT(*) AS no_of_items,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)                     AS perc_items,
       ROUND(SUM(a.total_revenue) * 100.0 / SUM(SUM(a.total_revenue)) OVER (), 1) AS perc_rev
FROM item_abc a
INNER JOIN item_xyz x
        ON a.item_id = x.item_id
GROUP BY abc_xyz_class
ORDER BY abc_xyz_class
""").df()

,abc_xyz_class,no_of_items,perc_items,perc_rev
0,AX,1119,36.7,68.0
1,AY,168,5.5,10.4
2,AZ,31,1.0,1.5
3,BX,712,23.4,11.4
4,BY,203,6.7,3.1
5,BZ,28,0.9,0.4
6,CX,533,17.5,3.5
7,CY,213,7.0,1.3
8,CZ,42,1.4,0.2


## 5. Save the matrix as a view
A **view** is a saved query: it stores the SQL, not the data, so it always reflects the latest `item_abc` and `item_xyz`.

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW v_abc_xyz_matrix AS
SELECT a.abc_rev || x.xyz_items AS segment,
       COUNT(*)                 AS total_items,
       SUM(a.total_revenue)     AS segment_revenue,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)                         AS perc_items,
       ROUND(SUM(a.total_revenue) * 100.0 / SUM(SUM(a.total_revenue)) OVER (), 2) AS perc_revenue
FROM item_abc a
INNER JOIN item_xyz x
        ON a.item_id = x.item_id
GROUP BY a.abc_rev || x.xyz_items
""")

In [ ]:
# Segments carrying more than 5% of revenue
con.sql("""
SELECT *
FROM v_abc_xyz_matrix
WHERE perc_revenue > 5
ORDER BY perc_revenue DESC
""").df()

,segment,total_items,segment_revenue,perc_items,perc_revenue
0,AX,1119,30637512.82,36.70,68.02
1,BX,712,5139691.39,23.35,11.41
2,AY,168,4693484.65,5.51,10.42


**Findings: ABC-XYZ**
- **AX: 1,119 items (37%) carry 68% of revenue.** High value and stable, these are the best candidates for automated forecasting with minimal planner touch.
- **AY + AZ: 199 items (6.5%) carry 12% of revenue.** High value but variable: this is where planner time and forecast-explanation effort pay off most.
- **CZ: 42 items, 0.2% of revenue.** Low value and erratic: simple rules (min/max) are good enough; heavy forecasting effort is wasted here.